# 🛒 İtopya İşlemci Fiyat Listesi Çekme (Web Scraping)

Bu proje, `itopya.com` üzerinden işlemcilerin isimlerini, güncel fiyatlarını ve ürün linklerini otomatik olarak çeken, kapsamlı bir Web Scraping uygulamasıdır.

Aşağıdaki hücreleri sırasıyla çalıştırarak sürecin nasıl işlediğini görebilirsiniz.

## 1. Adım: Siteye Bağlanma ve Durum Kontrolü

Hedef siteye bağlantı isteği atıyoruz.

In [ ]:
import requests
from bs4 import BeautifulSoup

# İtopya sitesine GET isteği atıyoruz
url = requests.get("https://www.itopya.com/islemci_k8")

# 200 kodu başarılı demektir
if url.status_code == 200:
    print("Siteden Veri Çekilebilir")
else:
    print("Siteden Veri Çekilemez")

## 2. Adım: Verileri Çekme ve Ekrana Yazdırma

İtopya'nın sayfa yapısına göre ürünler `<div class="product_product_List">` içinde listeleniyor. Bu ana divi bulup, içindeki her bir ürün bloğunu tek tek dolaşacağız.

In [ ]:
# HTML'i parse edilebilir (aranabilir) hale getir
soup = BeautifulSoup(url.content, "html.parser")

# Tüm ürünleri kapsayan en dış ana div'i bul
urun_listesi_div = soup.find("div", {"class": "product_product_List"})

if urun_listesi_div:
    # Ana div'in içindeki her bir ürün kartı (product-block) için döngü başlat
    for i in urun_listesi_div.find_all("div", {"class": "product-block"}):

        # Başlığı bul, en derindeki a etiketinden yazıyı çek ve boşlukları sil (strip)
        baslik_al = i.find("div", {"class": "product-block-top"}).h2.a.text.strip()

        # Fiyat etiketini bulup içindeki sayıyı al
        fiyat_al = i.find("span", {"class": "product-price"}).text.strip()

        # Linkin href özelliğini alıp, İtopya'nın ana domaini ile birleştir (Çünkü linkler /islemci... diye başlar)
        link = "https://www.itopya.com" + i.find("div", {"class": "product-image"}).a.get("href")

        # Ekrana düzgün ve okunaklı bir formatta yazdır
        print(f"\nÜrün Adı: {baslik_al}\nÜrün Fiyatı: {fiyat_al}\nÜrün Linki: {link}\n")
        print("-" * 50)

---

# 🔄 Programın Çalışma Mantığı ve Kodların Gerçek İşleyiş Sırası

### Aşama 1: Sitenin Tamamını Yüklemek
Python, `requests` ile bağlanır ve İtopya sayfasındaki fiyatlar, resimler, menüler dâhil tüm HTML iskeletini saniyeler içinde arka planda indirir.

### Aşama 2: Samanlıkta İğne Aramak (Ana Div'i Bulmak)
Biz menülerle veya alt kısımla (footer) ilgilenmiyoruz. Bize sadece vitrin lazım. Bu yüzden `soup.find("div", {"class": "product_product_List"})` komutu ile sitenin geri kalan tüm fazlalıklarını çöpe atıp, sadece ürünlerin bulunduğu kocaman vitrin kutusunu elimizde tutuyoruz.

### Aşama 3: Vitrindeki Ürünleri Ayırmak (Döngüye Sokmak)
Vitrin kutusu elimizde ama hepsi birbirine yapışık. `.find_all("div", {"class": "product-block"})` diyerek vitrindeki her bir kutuyu (her bir işlemci kartını) ayırıp sıraya diziyoruz.

### Aşama 4: Parçalara Ayırma ve Temizlik
Artık döngü çalışıyor ve her saniye elimize bir ürün kartı alıyoruz:
1. Kartın en üstüne çık (`product-block-top`), başlık etiketini bul, yazıyı kazı, başındaki-sonundaki fazla boşlukları sil (`.strip()`).
2. Fiyat etiketini (`product-price`) bul, sayıyı kazı.
3. Resim etiketinin (`product-image`) arkasında gizlenen tıklanabilir adresi (`href`) kopar ve önüne `https://www.itopya.com` ekleyerek tam ve tıklanabilir bir URL haline getir.

### Aşama 5: Ekrana Basma
Tüm temizlenen parçaları düzgün bir cümleye dönüştürüp ekrana basarız ve döngü bir sonraki ürüne geçer.